### Non-linear Option Pricing (MATHGR5400), Columbia University, Fall 2025
# Test Notebook

In this notebook, we demonstrate some typical uses of Python in the context of derivatives pricing. In particular, we provide some common numerical routines that you would need in the homework assigments. Please make sure the notebook runs to completion and produces expected numerical results and charts.

You need the following python packages installed on your computer to run this notebook:
<ul>
    <li>numpy</li>
    <li>scipy</li>
    <li>matplotlib</li>
    <li>torch</li>
</ul>

This is not an homework assignment. There is nothing for you to turn in. The notebook is provided solely for your reference.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import norm

### Black-Scholes Option Pricing Formula

In the Black-Scholes model, the price of an tradable asset $S_t$ is assumed to follow a geometric Brownian motion:

$$\frac{dS_t}{S_t}=\left(r-q\right)dt+\sigma dW_t$$

where $r$ is interest rate, $q$ is devidend yield, $\sigma$ is volatility, and $W_t$ is standard one-dimensional Brownian motion. Then the prices of vanilla call and put options are given by the following formulas:

$$\text{Price of Call}=S_0e^{-qT}N(d_1)-Ke^{-rT}N(d_2)\quad\text{and}\quad
\text{Price of Put}=Ke^{-rT}N(-d_2)-S_0e^{-qT}N(-d_1)$$

where $K$ is the strike and $T$ is the expiry of the option, and

$$d_1=\frac{1}{\sigma\sqrt{T}}\left[\log\frac{S_0}{K}+\left(r-q+\frac{1}{2}\sigma^2\right)T\right]
\quad\text{and}\quad d_2=d_1-\sigma\sqrt{T}$$

In [ ]:
def blackscholes_price(K, T, S0, vol, r=0, q=0, callput='call'):
    """Compute the call/put option price in the Black-Scholes model
    
    Parameters
    ----------
    K: scalar or array_like
        The strike of the option.
    T: scalar or array_like
        The maturity of the option, expressed in years (e.g. 0.25 for 3-month and 2 for 2 years)
    S0: scalar or array_like
        The current price of the underlying asset.
    vol: scalar or array_like
        The implied Black-Scholes volatility.
    r: scalar or array_like
        The annualized risk-free interest rate, continuously compounded.
    q: scalar or array_like
        The annualized continuous dividend yield.
    callput: str
        Must be either 'call' or 'put'.

    Returns
    -------
    price: scalar or array_like
        The price of the option.

    Examples
    --------
    >>> blackscholes_price(95, 0.25, 100, 0.2, r=0.05, callput='put')
    1.5342604771222823
    """
    F = S0*np.exp((r-q)*T)
    v = vol*np.sqrt(T)
    d1 = np.log(F/K)/v + 0.5*v
    d2 = d1 - v
    try:
        opttype = {'call':1, 'put':-1}[callput.lower()]
    except:
        raise ValueError('The value of callput must be either "call" or "put".')
    price = opttype*(F*norm.cdf(opttype*d1)-K*norm.cdf(opttype*d2))*np.exp(-r*T)
    return price

In [ ]:
# Compute the price of a European put option that expires in six months and has a strike price 95
# with current spot price 100, interest rate 5%, dividend yield 2%, Black-Scholes volatility 20%
blackscholes_price(K=95, T=0.5, S0=100, vol=0.2, r=0.05, q=0.02, callput='put')

In [ ]:
# Compute and plot the prices of European put options for an array of spot prices
spots = np.linspace(50, 150, 100)
prices = blackscholes_price(K=100, T=1, S0=spots, vol=0.2, r=0.05, q=0.02, callput='put')
fig, ax = plt.subplots(figsize=(6, 5.5))
ax.plot(spots, prices)
ax.grid(True)
ax.set_xlabel('Spot')
ax.set_ylabel('Price')

In [ ]:
# Compute the prices of European put options with a matrix of expiries and strikes, using numpy's broadcasting feature
Ks = np.array([90, 100, 110])
Ts = np.array([0.25, 0.5])
blackscholes_price(Ks, Ts[:, np.newaxis], S0=100, vol=0.2, r=0.05, q=0.02, callput='put')

### Monte Carlo Simulation in the Black-Scholes Model

In [ ]:
def blackscholes_mc(ts, n_paths, S0, vol, r, q):
    """Generate Monte-Carlo paths in Black-Scholes model.

    Parameters
    ----------
    ts: array_like
        The time steps of the simualtion
    n_paths: int
        the number of paths to simulate
    S0: scalar
        The spot price of the underlying security.
    vol: scalar
        The implied Black-Scholes volatility.
    r: scalar
        The annualized risk-free interest rate, continuously compounded.
    q: scalar
        The annualized continuous dividend yield.

    Returns
    -------
    paths: 2D ndarray of shape (len(ts), n_paths)
        The Monte-Carlo paths.
    """
    paths = np.full((len(ts), n_paths), np.nan, dtype=float)
    paths[0] = S0
    for i in range(len(ts)-1):
        dt = ts[i+1] - ts[i]
        dW = np.sqrt(dt)*np.random.randn(n_paths)
        paths[i+1] = paths[i] * np.exp((r-q-1/2*vol**2)*dt + vol*dW)
    return paths

In [ ]:
ts = np.linspace(0, 1, 253)
paths = blackscholes_mc(ts=ts, n_paths=10, S0=100, vol=0.2, r=0.05, q=0.02)
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(ts, paths, lw=1)
ax.grid(True)

In [ ]:
# Use Monte Carlo simulation to price an European put option
K = 95
T = 0.5
r = 0.05
q = 0.02
vol = 0.2
ts = np.linspace(0, T, int(round(T*252))+1) # daily timestep
paths = blackscholes_mc(ts=ts, n_paths=100000, S0=100, vol=vol, r=r, q=q)
np.mean(np.maximum(K-paths[-1], 0))*np.exp(-r*T)

### Black-Scholes Implied Volatility

In [ ]:
# all inputs must be scalar
def blackscholes_impv_scalar(K, T, S0, value, r=0, q=0, callput='call', tol=1e-6, maxiter=500):
    """Compute implied volatility in Black-Scholes model
    
    Parameters
    ----------
    K: scalar
        The strike of the option.
    T: scalar
        The maturity of the option.
    S0: scalar
        The current price of the underlying asset.
    value: scalar
        The value of the option
    callput: str
        Must be either 'call' or 'put'

    Returns
    -------
    vol: scalar
        The implied vol of the option.
    """
    if (K <= 0) or (T <= 0) or (S0 <= 0):
        return np.nan
    F = S0*np.exp((r-q)*T)
    K = K/F
    value = value*np.exp(r*T)/F
    try:
        opttype = {'call':1, 'put':-1}[callput.lower()]
    except:
        raise ValueError('The value of callput must be either "call" or "put".')
    # compute the time-value of the option
    value -= max(opttype * (1 - K), 0)
    if value < 0:
        return np.nan
    if (value == 0):
        return 0
    j = 1
    p = np.log(K)
    if K >= 1:
        x0 = np.sqrt(2 * p)
        x1 = x0 - (0.5 - K * norm.cdf(-x0) - value) * np.sqrt(2*np.pi)
        while (abs(x0 - x1) > tol*np.sqrt(T)) and (j < maxiter):
            x0 = x1
            d1 = -p/x1+0.5*x1
            x1 = x1 - (norm.cdf(d1) - K*norm.cdf(d1-x1)-value)*np.sqrt(2*np.pi)*np.exp(0.5*d1**2)
            j += 1
        return x1 / np.sqrt(T)
    else:
        x0 = np.sqrt(-2 * p)
        x1 = x0 - (0.5*K-norm.cdf(-x0)-value)*np.sqrt(2*np.pi)/K
        while (abs(x0-x1) > tol*np.sqrt(T)) and (j < maxiter):
            x0 = x1
            d1 = -p/x1+0.5*x1
            x1 = x1-(K*norm.cdf(x1-d1)-norm.cdf(-d1)-value)*np.sqrt(2*np.pi)*np.exp(0.5*d1**2)
            j += 1
        return x1 / np.sqrt(T)

# vectorized version
blackscholes_impv = np.vectorize(blackscholes_impv_scalar, excluded={'callput', 'tol', 'maxiter'})

In [ ]:
# Example
blackscholes_impv(K=95, T=0.25, S0=100, value=1.5, callput='put')

### Torch Examples

In this example, we use a neural network to estimate the price of a $T$-maturity European call option at an intermediate time $0<t<T$. The underlying asset price $S_t$ is assumed to follow a Black-Scholes model. The price of the call option at $t$ is then a (nonlinear) function of $S_{t}$. In particular, this price function $u(t,S_t)$ is the conditional expectation of the discounted final payoff at $T$ given the spot value at time $t$, i.e.,

$$u(t, S)=\mathbb{E}^{\mathbb{Q}}\left[\left.e^{-(T-t)}\left(S_T-K\right)^+\right\vert S_t=S\right].$$

A feed-forward neural network is used to estimate this conditional expectation. We simulate training paths in the Black-Scholes model and fit the network with the mean squared error (MSE) loss function to the simulated data.


Before proceeding please install the torch library as follows. 
```
$ pip install torch
```

In [ ]:
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
import time

# verify your torch version
torch.__version__

#### Preparing Training Data

In [ ]:
S0 = 100.
vol = 0.2
r = 0.05
q = 0.02
K = S0
ts = np.linspace(0, 1, 13)

In [ ]:
# simulate training paths
n_paths = 100000
paths = blackscholes_mc(ts=ts, n_paths=n_paths, S0=S0, vol=vol, r=r, q=q)

In [ ]:
# estimate price function at t = ts[j]
j = 6
X_train_full = paths[j]
# normalize the inputs
mX = np.mean(X_train_full)
sX = np.std(X_train_full)
X_train_full = ((X_train_full - mX) / sX)[:, np.newaxis]
Y_train_full = (np.maximum(paths[-1]-K, 0)*np.exp(-r*(ts[-1]-ts[j])))[:, np.newaxis]
# split the dataset to a training and a validation set
train_size = int(len(X_train_full)*0.75)
X_train = X_train_full[:train_size]
X_valid = X_train_full[train_size:]
Y_train = Y_train_full[:train_size]
Y_valid = Y_train_full[train_size:]

#### Creating and Compiling the Model

In [ ]:
learning_rate = 1e-3
hidden_size   = 20

# define the neural network. Use ReLU activation function and default weight initializer
model = nn.Sequential(
    nn.Linear(1, hidden_size), nn.ReLU(),
    nn.Linear(hidden_size, hidden_size), nn.ReLU(),
    nn.Linear(hidden_size, 1)
)
loss_fn   = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

#### Move Data to Tensors

In [ ]:
batch_size = 128

# Prepare training data to loop through
train_dataset = TensorDataset(
    torch.tensor(X_train, dtype=torch.float32), 
    torch.tensor(Y_train, dtype=torch.float32)
)
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

# Prepare validation data for evaluation after each epoch
X_valid_tensor = torch.tensor(X_valid, dtype=torch.float32) 
Y_valid_tensor = torch.tensor(Y_valid, dtype=torch.float32)

#### Training the Model

In [ ]:
epochs       = 50     # Max number of epochs of training
patience     = 6      # Number of epochs without imporovement for early stopping
min_delta    = 5e-3   # Minimum change in loss to be considered improvement
no_improv    = 0      # Remember epochs without improvements
last_epoch   = epochs # Keep track of epochs
restore_best = True   # Indicator to restore best performing model

# Keep track of best loss
best_valid_loss = np.inf

# Keep track of best model
best_model = model.state_dict()

# Train for max epochs 
for epoch in range(1, epochs+1):
    train_loss = 0.
    # Loop through training data to train
    for i, (X, Y) in enumerate(train_dataloader):
        loss  = loss_fn(model(X), Y)    
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        train_loss += loss.item()

    train_loss /= len(train_dataloader)
        
    # Evaluate validaiton for early stopping and monitoring
    with torch.no_grad():
        valid_loss = loss_fn(model(X_valid_tensor), Y_valid_tensor).item()
    
    # Check for improvement
    if valid_loss < best_valid_loss - min_delta:
        no_improv = 0
        best_valid_loss = valid_loss
        best_model = model.state_dict()
    else:
        no_improv += 1
    # Print epoch summary
    print(f"Epoch: {epoch:02}/50 Training Loss: {train_loss:.4f} - Validation Loss: {valid_loss:.4f} - Epochs without Improvement: {no_improv}")

    # Early stop on patience limit reached
    if no_improv >= patience:
        last_epoch = epoch + 1 
        print(f"Early Stopping at epoch {epoch}")
        break
        
if restore_best:
    model.load_state_dict(best_model)

#### Evaluating the Model

We compare the neural network predictions against the exact prices given by the Black-Scholes formula.

In [ ]:
model.eval()
fig, ax = plt.subplots(figsize=(6, 5.5))
x = np.linspace(*np.percentile(paths[j], [0.1, 99.9]), num=200)
s = torch.tensor((x-mX)/sX, dtype=torch.float32)
y_pred = model(s.view(-1, 1))[:, 0].detach().numpy()
y_true = blackscholes_price(K, ts[-1]-ts[j], x, vol=vol, r=r, q=q, callput='call')
ax.scatter(X_train[:200, 0]*sX+mX, Y_train[:200, 0], s=24, color='forestgreen', alpha=0.5, label='training data')
ax.plot(x, y_pred, lw=2, label='pred')
ax.plot(x, y_true, lw=2, label='true')
ax.legend(loc=0)
ax.set_title('Price of One-Year European Call at t={:.2f}'.format(ts[j]))
ax.set_xlabel('$S_t$')
ax.grid(True)